In [1]:
using SymPy

# Define symbols
z, f, e, ν₀, H, α = symbols("z f e nu0 H alpha", real=true, positive=true)
im = SymPy.im  # SymPy's imaginary unit

# Viscosity profile
nu = ν₀ + exp(α*z) + exp(-α*(z+H))

       alpha*z    -alpha*(H + z)
nu0 + e        + e              

In [2]:
# The ODE as described:
# f*u'(z) + ϵ^2*im*d/dz (nu(z) * u'(z)) = 0
# Let w(z) = u'(z):
# => f*w(z) + ϵ^2*im*diff(nu*w(z), z) = 0
# => diff(nu*w(z), z) = -f/(ϵ^2*im)*w(z)
# Expand: nu'*w + nu*w' + (f/(ϵ^2*im))w = 0
# Arrange as w'(z) + [nu'(z)/nu(z) + (f/(ϵ^2*im*nu(z)))] * w(z) = 0

# 1. The coefficient of w(z):
a = diff(nu, z)/nu + f/(e^2 * im * nu)
a = simplify(a)

         2 /     alpha*(H + 2*z)\        alpha*(H + z)
- alpha*e *\1 - e               / - I*f*e             
------------------------------------------------------
        2 //       alpha*z\  alpha*(H + z)    \       
       e *\\nu0 + e       /*e              + 1/       

In [5]:
# Pick the value of ξ in [-H, 0]:
ξᵢ     = -0.5
f_val  = 1.0
α_val  = 32.0
e_val  = 1.0
H_val  = 1.0
ν₀_val = 0.125
# Insert parameter values here:
a  = a.subs([(f, f_val), (ν₀, ν₀_val), (α, α_val), (e, e_val), (H, H_val)])

    /                      32.0*z                         64.0*z       \
1.0*\- 78962960182680.7*I*e       + 2.52681472584578e+15*e       - 32.0/
------------------------------------------------------------------------
                              / 32.0*z        \  32.0*z                 
             78962960182680.7*\e       + 0.125/*e       + 1             

In [6]:

# 2. General solution for w(z) (first-order linear ODE):
C1 = symbols("C1")
w = C1 * exp(-integrate(a, z))


                             /                                                 >
                            |                                                  >
                            |                                  64.0*z          >
                            |            2.52681472584578e+15*e                >
    - 1.26641655490942e-14* | ------------------------------------------------ >
                            |        32.0*z        64.0*z                      >
                            | 0.125*e       + 1.0*e       + 1.26641655490942e- >
                            |                                                  >
                           /                                                   >
C1*e                                                                           >

>                                /                                             >
>                               |                                              >
>                          

Evidently (GPT4.1) this is a known bug with SymPy.jl:

In [7]:

# 3. Integrate w(z) to find u(z):
C2 = symbols("C2")
u = C2 + integrate(w, z)


     /                                                                         >
    |                                                                          >
    |                          /                                               >
    |                         |                                                >
    |                         |                                  64.0*z        >
    |                         |            2.52681472584578e+15*e              >
    |  -1.26641655490942e-14* | ---------------------------------------------- >
    |                         |        32.0*z        64.0*z                    >
    |                         | 0.125*e       + 1.0*e       + 1.26641655490942 >
    |                         |                                                >
    |                        /                                                 >
C1* | e                                                                        >
    |                       

In [ ]:

# 4. Impose boundary condition u(-H) = 0:
# Plug z = -H and solve for C2:
u_mH = u.subs(z, -H_val)
eq = Eq(u_mH, 0)
C2sol = solve(eq, C2)[1]

# 5. Substitute C2 back into the expression for u(z):
u = u.subs(C2, C2sol)

println("General solution for u(z) with u(-H) = 0:")
display(u)